# 09 — Top-k ablation for the improved residual GATv2

This notebook performs a controlled graph-density experiment on the improved model:

\[
h_i^{final}
=
x_i^{DINO}
+
eta \Delta h_i^{GAT}
\]

followed by local patch matching and the score-level residual

\[
s_c^{final}
=
s_c^{CLS}
+
lpha s_c^{graph}.
\]

The only graph-construction hyperparameter varied across models is:

```text
top_k = 3
top_k = 5
top_k = 10
```

Everything else is kept fixed:

- same frozen DINOv2 cache;
- same train/validation split;
- same episodic seeds;
- same model initialization seed;
- same architecture;
- same optimizer;
- same number of train episodes;
- same validation episodes;
- same evaluation episodes.

Each top-k value receives its **own model trained from scratch**.

After training, all checkpoints are evaluated on the exact same validation episodes under:

- 5-way 5-shot;
- 5-way 1-shot;
- 10-way 5-shot;
- 10-way 1-shot.

The main comparison metrics are:

- accuracy;
- loss;
- gain over frozen CLS;
- paired 95% CI of the gain;
- CLS errors fixed;
- CLS-correct predictions broken;
- net fixed decisions;
- learned score residual `alpha`;
- learned patch residual `beta`;
- runtime.

## Why this experiment matters

The previous semantic-edge diagnostic showed that many of the selected top-10 edges had fairly low cosine similarity. This experiment tests whether a smaller semantic neighborhood prevents noisy cross-image message passing.

Do **not** use the test split for choosing `top_k`. This notebook intentionally uses validation.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

REPO_URL = "https://github.com/TomerBurman/CrossImagePatchGraph.git"
REPO_DIR = Path("/content/CrossImagePatchGraph_repo")
BRANCH_NAME = "experiment/max-mean-readout"

if not REPO_DIR.exists():
    !git clone -b "$BRANCH_NAME" "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" fetch
    !git -C "$REPO_DIR" checkout "$BRANCH_NAME"
    !git -C "$REPO_DIR" pull origin "$BRANCH_NAME"

%cd /content/CrossImagePatchGraph_repo
!pip install -q -r requirements.txt

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [ ]:
import gc
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from cross_image_glot.config import DEFAULT_PATHS
from cross_image_glot.models import (
    PatchGATv2Encoder,
    CrossImageGraphMatcher,
    BaselinePreservingResidualMatcher,
)
from cross_image_glot.storage import (
    restore_feature_splits,
    atomic_json_save,
)
from cross_image_glot.data import (
    MiniImageNetFeatureDataset,
    FewShotFeatureEpisodeDataset,
)
from cross_image_glot.graph_builder import (
    ClassConditionedPatchGraphBuilder,
)
from cross_image_glot.baselines import frozen_baseline_episode
from cross_image_glot.training import (
    evaluate_residual_dataset,
    evaluate_residual_feature_episode,
    load_training_checkpoint,
    make_checkpoint,
    save_checkpoint_atomic,
    save_history,
    train_residual_epoch,
)

paths = DEFAULT_PATHS
paths.ensure_directories()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Drive root:", paths.drive_root)


## Experimental model components

These are the same components used in the improved residual-GAT experiment:

- `DinoResidualGraphEncoder`
- `MaxMeanPatchReadout`
- `frozen_patch_match_episode`

They are defined inline so this notebook remains self-contained.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch import nn


@dataclass
class PatchMatchReadoutOutput:
    """One score per candidate graph."""
    scores: torch.Tensor
    raw_similarities: torch.Tensor


class DinoResidualGraphEncoder(nn.Module):
    """
    Preserve the original frozen DINOv2 patch representation and let a graph
    encoder learn only a residual correction.

    graph.x[:, :dino_dim] must contain the original DINOv2 patch token.

        base_encoder: [nodes, input_dim] -> [nodes, base_output_dim]
        delta head:   [nodes, base_output_dim] -> [nodes, dino_dim]
        output:       DINO + beta * delta

    The correction is norm-matched to the original DINO token so beta has a
    stable interpretation. beta=0.05 starts at roughly a 5% correction.
    """

    def __init__(
        self,
        base_encoder: nn.Module,
        base_output_dim: int,
        dino_dim: int = 384,
        initial_patch_residual_scale: float = 0.05,
        norm_match_delta: bool = True,
    ) -> None:
        super().__init__()
        self.base_encoder = base_encoder
        self.base_output_dim = base_output_dim
        self.dino_dim = dino_dim
        self.hidden_dim = dino_dim
        self.norm_match_delta = norm_match_delta

        self.delta_projection = nn.Linear(base_output_dim, dino_dim)
        self.patch_residual_scale = nn.Parameter(
            torch.tensor(float(initial_patch_residual_scale), dtype=torch.float32)
        )

    def forward(self, graph):
        if graph.x.shape[-1] < self.dino_dim:
            raise ValueError(
                f"graph.x has {graph.x.shape[-1]} features, but dino_dim={self.dino_dim}."
            )

        original_dino = graph.x[:, : self.dino_dim].to(torch.float32)
        graph_hidden = self.base_encoder(graph)
        delta = self.delta_projection(graph_hidden)

        if self.norm_match_delta:
            delta = F.normalize(delta, p=2, dim=-1)
            original_norm = (
                original_dino.norm(p=2, dim=-1, keepdim=True)
                .detach()
                .clamp_min(1e-6)
            )
            delta = delta * original_norm

        return original_dino + self.patch_residual_scale * delta


class MaxMeanPatchReadout(nn.Module):
    """
    Patch-to-patch scoring instead of mean-pooling all patches first.

    Default candidate score:

        mean over support images j [
            mean over query patches i [
                max over support patches p cosine(q_i, s_{j,p})
            ]
        ]

    This preserves local correspondences and gives each support image equal
    weight. `classwide_max` instead matches each query patch against all support
    patches of the candidate class jointly.
    """

    def __init__(
        self,
        temperature: float = 0.1,
        support_reduction: str = "mean_image",
    ) -> None:
        super().__init__()
        if temperature <= 0:
            raise ValueError("temperature must be positive.")
        if support_reduction not in {"mean_image", "classwide_max"}:
            raise ValueError(
                "support_reduction must be 'mean_image' or 'classwide_max'."
            )
        self.temperature = float(temperature)
        self.support_reduction = support_reduction

    def _single_graph_score(self, nodes, image_ids):
        query = nodes[image_ids == 0]
        if query.numel() == 0:
            raise ValueError("Candidate graph has no query patches.")
        query = F.normalize(query, p=2, dim=-1)

        support_ids = torch.unique(image_ids[image_ids > 0], sorted=True)
        if support_ids.numel() == 0:
            raise ValueError("Candidate graph has no support patches.")

        if self.support_reduction == "classwide_max":
            support = F.normalize(nodes[image_ids > 0], p=2, dim=-1)
            similarity = query @ support.T
            return similarity.max(dim=1).values.mean()

        per_image_scores = []
        for support_id in support_ids:
            support = F.normalize(nodes[image_ids == support_id], p=2, dim=-1)
            similarity = query @ support.T
            per_image_scores.append(similarity.max(dim=1).values.mean())
        return torch.stack(per_image_scores).mean()

    def forward(self, refined_nodes: torch.Tensor, graph_batch):
        if not hasattr(graph_batch, "image_id"):
            raise AttributeError("graph_batch must contain image_id metadata.")

        image_ids = graph_batch.image_id.reshape(-1).long()
        if hasattr(graph_batch, "batch"):
            graph_ids = graph_batch.batch.reshape(-1).long()
            num_graphs = int(graph_batch.num_graphs)
        else:
            graph_ids = torch.zeros(
                refined_nodes.shape[0], dtype=torch.long, device=refined_nodes.device
            )
            num_graphs = 1

        raw_scores = []
        for graph_id in range(num_graphs):
            mask = graph_ids == graph_id
            raw_scores.append(
                self._single_graph_score(refined_nodes[mask], image_ids[mask])
            )

        raw_scores = torch.stack(raw_scores)
        return PatchMatchReadoutOutput(
            scores=raw_scores / self.temperature,
            raw_similarities=raw_scores,
        )


@torch.inference_mode()
def frozen_patch_match_episode(
    episode: dict,
    device: torch.device,
    temperature: float = 0.1,
    support_reduction: str = "mean_image",
) -> tuple[torch.Tensor, torch.Tensor]:
    """Non-GNN local DINO patch-matching baseline."""
    support = episode["support_patches"].to(device=device, dtype=torch.float32)
    query = episode["query_patches"].to(device=device, dtype=torch.float32)
    labels = episode["query_labels"].to(device=device, dtype=torch.long)

    n_way, k_shot, _, _ = support.shape
    queries_per_class = query.shape[1]
    support = F.normalize(support, p=2, dim=-1)
    query = F.normalize(query, p=2, dim=-1)

    logits_rows = []
    targets = []

    for query_class_position in range(n_way):
        for query_position in range(queries_per_class):
            q = query[query_class_position, query_position]
            candidate_scores = []

            for candidate_id in range(n_way):
                candidate_support = support[candidate_id]

                if support_reduction == "classwide_max":
                    flat_support = candidate_support.reshape(
                        -1, candidate_support.shape[-1]
                    )
                    similarity = q @ flat_support.T
                    raw_score = similarity.max(dim=1).values.mean()
                elif support_reduction == "mean_image":
                    per_image_scores = []
                    for support_index in range(k_shot):
                        similarity = q @ candidate_support[support_index].T
                        per_image_scores.append(
                            similarity.max(dim=1).values.mean()
                        )
                    raw_score = torch.stack(per_image_scores).mean()
                else:
                    raise ValueError(
                        "support_reduction must be 'mean_image' or 'classwide_max'."
                    )

                candidate_scores.append(raw_score / temperature)

            logits_rows.append(torch.stack(candidate_scores))
            targets.append(labels[query_class_position, query_position])

    return torch.stack(logits_rows), torch.stack(targets)


## 1. Ablation configuration

A few deliberate changes from the first improved-model notebook:

- validation episodes per epoch are increased from 20 to **50** to reduce checkpoint-selection noise;
- early-stopping patience is increased from 3 to **5**;
- all top-k models are initialized using the same model seed.

The training protocol itself remains 5-way 5-shot. The other protocols are cross-shot/cross-way evaluations of those same trained checkpoints.


In [ ]:
TOP_K_VALUES = [3, 5, 10]

BASE_CONFIG = {
    "n_way": 5,
    "k_shot": 5,

    "input_dim": 387,
    "dino_dim": 384,
    "hidden_dim": 256,
    "num_layers": 2,
    "attention_heads": 4,
    "edge_dim": 5,
    "dropout": 0.1,

    "min_similarity": None,

    "graph_temperature": 0.1,
    "cls_temperature": 0.1,

    "initial_patch_residual_scale": 0.05,
    "initial_score_residual_scale": 0.05,
    "patch_support_reduction": "mean_image",

    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "graph_microbatch_size": 2,

    "train_queries_per_class": 1,
    "eval_queries_per_class": 15,

    "train_seed": 123,
    "val_seed": 456,
    "model_init_seed": 2026,

    "train_num_episodes": 1000,
    "val_num_episodes": 600,

    "num_epochs": 10,
    "train_episodes_per_epoch": 100,

    # More stable model selection than the earlier 20-episode check.
    "validation_episodes_per_epoch": 50,
    "final_validation_episodes": 100,
    "early_stopping_patience": 5,

    "max_cached_shards": 6,
}

# Training behavior:
# - If a best checkpoint already exists, set this True to avoid retraining it.
# - If False, the notebook resumes from latest.pt when available.
SKIP_IF_BEST_EXISTS = False
RESUME_IF_LATEST_EXISTS = True

# Final paired evaluation.
EVAL_NUM_EPISODES = 100
EVAL_QUERIES_PER_CLASS = 15
EVAL_SEED = 40_000

EVAL_PROTOCOLS = [
    ("5W5S", 5, 5),
    ("5W1S", 5, 1),
    ("10W5S", 10, 5),
    ("10W1S", 10, 1),
]

print("top-k values:", TOP_K_VALUES)
print(json.dumps(BASE_CONFIG, indent=2))


## 2. Restore frozen DINO train/validation features

No DINOv2 forward passes occur in this notebook.

All models see exactly the same cached frozen DINO features.


In [ ]:
drive.mount("/content/drive", force_remount=True)

restore_feature_splits(
    ["train", "val"],
    paths.drive_feature_dir,
    paths.local_feature_dir,
)

train_features = MiniImageNetFeatureDataset(
    paths.local_feature_dir,
    "train",
    max_cached_shards=BASE_CONFIG["max_cached_shards"],
)

val_features = MiniImageNetFeatureDataset(
    paths.local_feature_dir,
    "val",
    max_cached_shards=BASE_CONFIG["max_cached_shards"],
)

print("Train images:", len(train_features))
print("Validation images:", len(val_features))
print("Patch grid:", train_features.metadata["grid_size"])


## 3. Utility functions

`reset_random_state()` is important for a fair top-k ablation.

Before constructing each model we reset Python, NumPy and PyTorch RNGs to the same seed. Therefore differences are not intentionally caused by different random parameter initialization.


In [ ]:
def reset_random_state(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ci95(values):
    values = np.asarray(values, dtype=float)

    if len(values) <= 1:
        return float("nan")

    return (
        1.96
        * values.std(ddof=1)
        / math.sqrt(len(values))
    )


def experiment_name(top_k):
    return (
        "residual_gatv2_dino_patchmatch_"
        f"topk{top_k}_5way5shot"
    )


def make_config(top_k):
    config = dict(BASE_CONFIG)
    config["top_k"] = int(top_k)
    config["experiment_name"] = experiment_name(top_k)
    return config


def make_train_episodes(config):
    # Fresh dataset object for every top-k model.
    # Same seed => same episode sampling distribution.
    return FewShotFeatureEpisodeDataset(
        train_features,
        config["n_way"],
        config["k_shot"],
        config["train_queries_per_class"],
        num_episodes=config["train_num_episodes"],
        seed=config["train_seed"],
        vary_by_epoch=True,
    )


def make_validation_episodes(config):
    return FewShotFeatureEpisodeDataset(
        val_features,
        config["n_way"],
        config["k_shot"],
        config["eval_queries_per_class"],
        num_episodes=config["val_num_episodes"],
        seed=config["val_seed"],
        vary_by_epoch=False,
    )


def make_graph_builder(top_k):
    return ClassConditionedPatchGraphBuilder(
        grid_size=tuple(
            train_features.metadata["grid_size"]
        ),
        top_k=int(top_k),
        min_similarity=BASE_CONFIG["min_similarity"],
        graph_dtype=torch.float32,
        similarity_device=device,
    )


def build_model(config):
    reset_random_state(config["model_init_seed"])

    base_gat = PatchGATv2Encoder(
        input_dim=config["input_dim"],
        hidden_dim=config["hidden_dim"],
        num_layers=config["num_layers"],
        heads=config["attention_heads"],
        edge_dim=config["edge_dim"],
        dropout=config["dropout"],
    )

    encoder = DinoResidualGraphEncoder(
        base_encoder=base_gat,
        base_output_dim=config["hidden_dim"],
        dino_dim=config["dino_dim"],
        initial_patch_residual_scale=config[
            "initial_patch_residual_scale"
        ],
    )

    readout = MaxMeanPatchReadout(
        temperature=config["graph_temperature"],
        support_reduction=config[
            "patch_support_reduction"
        ],
    )

    graph_matcher = CrossImageGraphMatcher(
        encoder=encoder,
        readout=readout,
    )

    model = BaselinePreservingResidualMatcher(
        graph_matcher,
        config["initial_score_residual_scale"],
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    return model, encoder, optimizer


## 4. Compare the semantic-edge distributions before training

This does not determine which `top_k` is best by itself.

It simply verifies the expected tradeoff:

- smaller `k` → fewer, generally stronger semantic edges;
- larger `k` → more message-passing routes, but potentially noisier ones.

The diagnostic also separates **true candidate graphs** from **wrong candidate graphs**. This is more informative than pooling every semantic edge together.


In [ ]:
def semantic_edge_stats(
    builder,
    episode,
    max_query_candidate_graphs=100,
):
    all_values = []
    correct_values = []
    wrong_values = []

    built = 0
    n_way = episode["support_patches"].shape[0]
    q_per_class = episode["query_patches"].shape[1]

    for query_class in range(n_way):
        for query_pos in range(q_per_class):
            for candidate in range(n_way):
                graph = builder.build_graph(
                    query_patches=episode[
                        "query_patches"
                    ][query_class, query_pos],
                    support_patches=episode[
                        "support_patches"
                    ][candidate],
                    candidate_id=candidate,
                )

                semantic_mask = (
                    graph.edge_type.reshape(-1)
                    == builder.SEMANTIC_EDGE
                )

                values = (
                    graph.edge_attr[
                        semantic_mask,
                        0,
                    ]
                    .float()
                    .cpu()
                )

                all_values.append(values)

                if candidate == query_class:
                    correct_values.append(values)
                else:
                    wrong_values.append(values)

                built += 1

                if built >= max_query_candidate_graphs:
                    break

            if built >= max_query_candidate_graphs:
                break

        if built >= max_query_candidate_graphs:
            break

    def summarize(parts):
        x = torch.cat(parts)

        q = torch.quantile(
            x,
            torch.tensor(
                [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
            ),
        )

        return {
            "mean": float(x.mean()),
            "median": float(q[3]),
            "quantiles": [float(v) for v in q],
            "num_edges": int(x.numel()),
        }

    return {
        "all": summarize(all_values),
        "correct_candidate": summarize(correct_values),
        "wrong_candidate": summarize(wrong_values),
    }


diagnostic_episode = FewShotFeatureEpisodeDataset(
    val_features,
    n_way=5,
    k_shot=5,
    queries_per_class=15,
    num_episodes=1,
    seed=EVAL_SEED,
    vary_by_epoch=False,
)[0]

edge_diagnostics = {}

for top_k in TOP_K_VALUES:
    print(f"\nTop-k = {top_k}")

    builder = make_graph_builder(top_k)

    stats = semantic_edge_stats(
        builder,
        diagnostic_episode,
        max_query_candidate_graphs=100,
    )

    edge_diagnostics[str(top_k)] = stats
    print(json.dumps(stats, indent=2))


## 5. Train every top-k model

Each model receives its own checkpoint directory:

```text
checkpoints/
    residual_gatv2_dino_patchmatch_topk3_5way5shot/
    residual_gatv2_dino_patchmatch_topk5_5way5shot/
    residual_gatv2_dino_patchmatch_topk10_5way5shot/
```

The models are all trained as 5-way 5-shot models.

Checkpoint selection is based only on validation accuracy.


In [ ]:
training_summaries = []
trained_checkpoint_paths = {}

for top_k in TOP_K_VALUES:
    print("\n" + "=" * 90)
    print(f"TRAINING TOP-K = {top_k}")
    print("=" * 90)

    config = make_config(top_k)

    checkpoint_dir = (
        paths.drive_checkpoint_dir
        / config["experiment_name"]
    )
    result_dir = (
        paths.drive_results_dir
        / config["experiment_name"]
    )

    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True,
    )
    result_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    latest_path = checkpoint_dir / "latest.pt"
    best_path = checkpoint_dir / "best.pt"

    trained_checkpoint_paths[top_k] = best_path

    if SKIP_IF_BEST_EXISTS and best_path.exists():
        print(
            "Best checkpoint already exists; "
            "skipping training:",
            best_path,
        )

        checkpoint = torch.load(
            best_path,
            map_location="cpu",
            weights_only=False,
        )

        training_summaries.append(
            {
                "top_k": top_k,
                "experiment_name": config[
                    "experiment_name"
                ],
                "best_validation_accuracy": checkpoint.get(
                    "best_validation_accuracy",
                    np.nan,
                ),
                "best_epoch": checkpoint.get(
                    "epoch",
                    np.nan,
                ),
                "status": "existing_checkpoint",
            }
        )
        continue

    train_episodes = make_train_episodes(config)
    val_episodes = make_validation_episodes(config)
    builder = make_graph_builder(top_k)

    model, encoder, optimizer = build_model(config)

    print(
        "Initial alpha:",
        float(
            model.residual_scale
            .detach()
            .cpu()
        ),
    )
    print(
        "Initial beta:",
        float(
            encoder.patch_residual_scale
            .detach()
            .cpu()
        ),
    )

    history = []
    start_epoch = 0
    best_accuracy = float("-inf")
    without_improvement = 0

    if (
        RESUME_IF_LATEST_EXISTS
        and latest_path.exists()
    ):
        state = load_training_checkpoint(
            latest_path,
            model,
            optimizer,
            device,
        )

        start_epoch = state["epoch"] + 1
        best_accuracy = state[
            "best_validation_accuracy"
        ]
        without_improvement = state[
            "epochs_without_improvement"
        ]
        history = state.get("history", [])

        print(
            "Resuming from epoch",
            state["epoch"] + 1,
        )

    for epoch in range(
        start_epoch,
        config["num_epochs"],
    ):
        print(
            f"\nTop-k {top_k} | "
            f"Epoch {epoch + 1}/{config['num_epochs']}"
        )

        train_metrics = train_residual_epoch(
            model,
            optimizer,
            builder,
            train_episodes,
            device,
            epoch,
            config["train_episodes_per_epoch"],
            config["graph_microbatch_size"],
            config["cls_temperature"],
            log_interval=10,
        )

        validation_metrics = evaluate_residual_dataset(
            model,
            builder,
            val_episodes,
            device,
            config[
                "validation_episodes_per_epoch"
            ],
            config["graph_microbatch_size"],
            config["cls_temperature"],
            log_interval=10,
        )

        record = {
            "top_k": top_k,
            "epoch": epoch,
            "train_loss": train_metrics.loss,
            "train_accuracy": train_metrics.accuracy,
            "validation_loss": (
                validation_metrics.loss
            ),
            "validation_accuracy": (
                validation_metrics.accuracy
            ),
            "score_alpha": float(
                model.residual_scale
                .detach()
                .cpu()
            ),
            "patch_beta": float(
                encoder.patch_residual_scale
                .detach()
                .cpu()
            ),
        }

        history.append(record)

        improved = (
            validation_metrics.accuracy
            > best_accuracy
        )

        if improved:
            best_accuracy = (
                validation_metrics.accuracy
            )
            without_improvement = 0
        else:
            without_improvement += 1

        checkpoint = make_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            best_validation_accuracy=best_accuracy,
            epochs_without_improvement=without_improvement,
            history=history,
            configuration=config,
        )

        save_checkpoint_atomic(
            checkpoint,
            latest_path,
        )

        if improved:
            save_checkpoint_atomic(
                checkpoint,
                best_path,
            )

        save_history(
            history,
            result_dir,
        )

        print(record)
        print(
            "best validation accuracy:",
            best_accuracy,
        )

        if (
            without_improvement
            >= config[
                "early_stopping_patience"
            ]
        ):
            print("Early stopping.")
            break

    if not best_path.exists():
        raise RuntimeError(
            f"No best checkpoint created for top_k={top_k}."
        )

    best_checkpoint = torch.load(
        best_path,
        map_location="cpu",
        weights_only=False,
    )

    training_summaries.append(
        {
            "top_k": top_k,
            "experiment_name": config[
                "experiment_name"
            ],
            "best_validation_accuracy": (
                best_checkpoint.get(
                    "best_validation_accuracy",
                    np.nan,
                )
            ),
            "best_epoch": best_checkpoint.get(
                "epoch",
                np.nan,
            ),
            "status": "trained",
        }
    )

    # Release GPU memory before constructing the next model.
    del model, encoder, optimizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


training_summary_df = pd.DataFrame(
    training_summaries
)

display(training_summary_df)


## 6. Plot training curves

These curves help distinguish a genuinely better graph density from a lucky final checkpoint.

Look for:

- consistently lower validation loss;
- consistently higher validation accuracy;
- whether `alpha` and `beta` behave differently as graph density changes.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for top_k in TOP_K_VALUES:
    history_path = (
        paths.drive_results_dir
        / experiment_name(top_k)
        / "history.json"
    )

    if not history_path.exists():
        continue

    history = json.loads(
        history_path.read_text()
    )

    epochs = [
        row["epoch"] + 1
        for row in history
    ]
    validation_accuracy = [
        100.0 * row["validation_accuracy"]
        for row in history
    ]

    ax.plot(
        epochs,
        validation_accuracy,
        marker="o",
        label=f"top_k={top_k}",
    )

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation accuracy (%)")
ax.set_title(
    "Top-k ablation — validation accuracy"
)
ax.legend()
ax.grid(alpha=0.25)
plt.show()


fig, ax = plt.subplots(figsize=(10, 5))

for top_k in TOP_K_VALUES:
    history_path = (
        paths.drive_results_dir
        / experiment_name(top_k)
        / "history.json"
    )

    if not history_path.exists():
        continue

    history = json.loads(
        history_path.read_text()
    )

    epochs = [
        row["epoch"] + 1
        for row in history
    ]
    validation_loss = [
        row["validation_loss"]
        for row in history
    ]

    ax.plot(
        epochs,
        validation_loss,
        marker="o",
        label=f"top_k={top_k}",
    )

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation cross-entropy loss")
ax.set_title(
    "Top-k ablation — validation loss"
)
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## 7. Evaluation helpers

The evaluation is paired:

- every top-k checkpoint;
- frozen CLS;
- frozen mean-patch;
- frozen local-patch matching;

all see exactly the same episodes.

This allows us to count exactly which CLS decisions were fixed or broken.


In [ ]:
@torch.inference_mode()
def evaluate_one_topk_protocol(
    model,
    builder,
    config,
    episode_dataset,
    protocol_name,
    num_episodes,
):
    rows = []

    cls_episode_accuracy = []
    model_episode_accuracy = []

    total_fixed = 0
    total_broken = 0

    start_time = time.perf_counter()

    for episode_index in range(num_episodes):
        episode = episode_dataset[
            episode_index
        ]

        cls_logits, targets = (
            frozen_baseline_episode(
                episode,
                "cls",
                device,
                temperature=config[
                    "cls_temperature"
                ],
            )
        )

        mean_logits, mean_targets = (
            frozen_baseline_episode(
                episode,
                "mean_patch",
                device,
                temperature=config[
                    "graph_temperature"
                ],
            )
        )

        local_logits, local_targets = (
            frozen_patch_match_episode(
                episode,
                device,
                temperature=config[
                    "graph_temperature"
                ],
                support_reduction=config[
                    "patch_support_reduction"
                ],
            )
        )

        result = evaluate_residual_feature_episode(
            model,
            builder,
            episode,
            device,
            graph_microbatch_size=config[
                "graph_microbatch_size"
            ],
            cls_temperature=config[
                "cls_temperature"
            ],
        )

        model_logits = result.logits.to(device)

        assert torch.equal(
            targets,
            mean_targets,
        )
        assert torch.equal(
            targets,
            local_targets,
        )

        cls_pred = cls_logits.argmax(dim=-1)
        mean_pred = mean_logits.argmax(dim=-1)
        local_pred = local_logits.argmax(dim=-1)
        model_pred = model_logits.argmax(dim=-1)

        cls_correct = cls_pred.eq(targets)
        model_correct = model_pred.eq(targets)

        total_fixed += int(
            (
                (~cls_correct)
                & model_correct
            ).sum()
        )

        total_broken += int(
            (
                cls_correct
                & (~model_correct)
            ).sum()
        )

        cls_acc = float(
            cls_correct.float().mean()
        )
        mean_acc = float(
            mean_pred.eq(targets)
            .float()
            .mean()
        )
        local_acc = float(
            local_pred.eq(targets)
            .float()
            .mean()
        )
        model_acc = float(
            model_correct.float().mean()
        )

        cls_episode_accuracy.append(cls_acc)
        model_episode_accuracy.append(model_acc)

        entries = [
            (
                "Frozen CLS",
                cls_logits,
                cls_acc,
            ),
            (
                "Frozen mean-patch",
                mean_logits,
                mean_acc,
            ),
            (
                "Frozen local patch match",
                local_logits,
                local_acc,
            ),
            (
                f"Improved GATv2 top_k={config['top_k']}",
                model_logits,
                model_acc,
            ),
        ]

        for model_name, logits, accuracy in entries:
            rows.append(
                {
                    "protocol": protocol_name,
                    "episode_index": episode_index,
                    "model": model_name,
                    "loss": float(
                        F.cross_entropy(
                            logits,
                            targets,
                        )
                    ),
                    "accuracy": accuracy,
                }
            )

        if (
            (episode_index + 1) % 10
            == 0
        ):
            print(
                f"top_k={config['top_k']} | "
                f"{protocol_name}: "
                f"{episode_index + 1}/{num_episodes}"
            )

    runtime = (
        time.perf_counter()
        - start_time
    )

    frame = pd.DataFrame(rows)

    summary = (
        frame.groupby(
            ["protocol", "model"]
        )
        .agg(
            loss=("loss", "mean"),
            accuracy=("accuracy", "mean"),
            episode_std=("accuracy", "std"),
            num_episodes=("accuracy", "count"),
        )
        .reset_index()
    )

    summary["accuracy_percent"] = (
        100.0 * summary["accuracy"]
    )

    summary["episode_accuracy_ci95_pp"] = (
        100.0
        * 1.96
        * summary["episode_std"]
        / np.sqrt(
            summary["num_episodes"]
        )
    )

    cls_accuracy = float(
        np.mean(cls_episode_accuracy)
    )

    summary["delta_vs_cls_pp"] = (
        100.0
        * (
            summary["accuracy"]
            - cls_accuracy
        )
    )

    paired_delta = (
        np.asarray(
            model_episode_accuracy
        )
        - np.asarray(
            cls_episode_accuracy
        )
    )

    graph_name = (
        f"Improved GATv2 "
        f"top_k={config['top_k']}"
    )

    graph_mask = (
        summary["model"]
        == graph_name
    )

    summary.loc[
        graph_mask,
        "paired_gain_ci95_pp",
    ] = 100.0 * ci95(paired_delta)

    summary.loc[
        graph_mask,
        "fixed_by_model",
    ] = total_fixed

    summary.loc[
        graph_mask,
        "broken_by_model",
    ] = total_broken

    summary.loc[
        graph_mask,
        "net_fixed",
    ] = (
        total_fixed
        - total_broken
    )

    summary.loc[
        graph_mask,
        "runtime_seconds",
    ] = runtime

    return summary


## 8. Load each best checkpoint and evaluate all protocols

The three top-k checkpoints are evaluated using the same protocol episode seeds.

This means differences between top-k models are paired at the episode level.


In [ ]:
evaluation_frames = []

for top_k in TOP_K_VALUES:
    print("\n" + "=" * 90)
    print(f"EVALUATING TOP-K = {top_k}")
    print("=" * 90)

    config = make_config(top_k)

    best_path = (
        paths.drive_checkpoint_dir
        / config["experiment_name"]
        / "best.pt"
    )

    if not best_path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint for top_k={top_k}: "
            f"{best_path}"
        )

    model, encoder, _ = build_model(config)

    checkpoint = torch.load(
        best_path,
        map_location="cpu",
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )
    model.to(device)
    model.eval()

    builder = make_graph_builder(top_k)

    learned_alpha = float(
        model.residual_scale
        .detach()
        .cpu()
    )

    learned_beta = float(
        encoder.patch_residual_scale
        .detach()
        .cpu()
    )

    print("Checkpoint:", best_path)
    print("Best epoch:", checkpoint.get("epoch"))
    print(
        "Best validation accuracy:",
        checkpoint.get(
            "best_validation_accuracy"
        ),
    )
    print("alpha:", learned_alpha)
    print("beta:", learned_beta)

    for (
        protocol_name,
        n_way,
        k_shot,
    ) in EVAL_PROTOCOLS:
        eval_episodes = (
            FewShotFeatureEpisodeDataset(
                val_features,
                n_way=n_way,
                k_shot=k_shot,
                queries_per_class=(
                    EVAL_QUERIES_PER_CLASS
                ),
                num_episodes=EVAL_NUM_EPISODES,
                seed=EVAL_SEED,
                vary_by_epoch=False,
            )
        )

        summary = evaluate_one_topk_protocol(
            model=model,
            builder=builder,
            config=config,
            episode_dataset=eval_episodes,
            protocol_name=protocol_name,
            num_episodes=EVAL_NUM_EPISODES,
        )

        graph_name = (
            f"Improved GATv2 "
            f"top_k={top_k}"
        )

        graph_mask = (
            summary["model"]
            == graph_name
        )

        summary.loc[
            graph_mask,
            "top_k",
        ] = top_k

        summary.loc[
            graph_mask,
            "score_alpha",
        ] = learned_alpha

        summary.loc[
            graph_mask,
            "patch_beta",
        ] = learned_beta

        summary.loc[
            graph_mask,
            "best_epoch",
        ] = checkpoint.get(
            "epoch",
            np.nan,
        )

        summary.loc[
            graph_mask,
            "checkpoint_validation_accuracy",
        ] = checkpoint.get(
            "best_validation_accuracy",
            np.nan,
        )

        evaluation_frames.append(summary)

    del model, encoder

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


raw_evaluation_df = pd.concat(
    evaluation_frames,
    ignore_index=True,
)


## 9. Clean comparison table

The baseline rows are repeated once for each top-k checkpoint because each top-k evaluation function computes them on the same episodes.

This cell deduplicates them and produces the final table.


In [ ]:
baseline_names = {
    "Frozen CLS",
    "Frozen mean-patch",
    "Frozen local patch match",
}

baseline_df = (
    raw_evaluation_df[
        raw_evaluation_df["model"].isin(
            baseline_names
        )
    ]
    .drop_duplicates(
        subset=[
            "protocol",
            "model",
        ]
    )
)

graph_df = raw_evaluation_df[
    ~raw_evaluation_df["model"].isin(
        baseline_names
    )
]

comparison_df = pd.concat(
    [baseline_df, graph_df],
    ignore_index=True,
)

display_columns = [
    "protocol",
    "model",
    "top_k",
    "loss",
    "accuracy_percent",
    "episode_accuracy_ci95_pp",
    "delta_vs_cls_pp",
    "paired_gain_ci95_pp",
    "fixed_by_model",
    "broken_by_model",
    "net_fixed",
    "score_alpha",
    "patch_beta",
    "best_epoch",
    "checkpoint_validation_accuracy",
    "runtime_seconds",
]

display(
    comparison_df[
        [
            column
            for column in display_columns
            if column in comparison_df.columns
        ]
    ]
    .sort_values(
        [
            "protocol",
            "accuracy_percent",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)


## 10. Direct top-k ranking

This table removes the frozen baselines and ranks only the three trained graph models.

For this ablation, prioritize:

1. paired gain over CLS;
2. `net_fixed`;
3. consistency across 1-shot protocols;
4. validation loss;
5. only then tiny differences in 5W5S accuracy.

A top-k setting that is best only on 5W5S but worse on 5W1S/10W1S is less interesting for the current project hypothesis.


In [ ]:
topk_only_df = (
    comparison_df[
        comparison_df["model"].str.startswith(
            "Improved GATv2"
        )
    ]
    .copy()
)

display(
    topk_only_df[
        [
            "protocol",
            "top_k",
            "accuracy_percent",
            "delta_vs_cls_pp",
            "paired_gain_ci95_pp",
            "fixed_by_model",
            "broken_by_model",
            "net_fixed",
            "score_alpha",
            "patch_beta",
            "loss",
        ]
    ]
    .sort_values(
        [
            "protocol",
            "accuracy_percent",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)


## 11. Plot gain over CLS by top-k


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for top_k in TOP_K_VALUES:
    subset = (
        topk_only_df[
            topk_only_df["top_k"]
            == top_k
        ]
        .set_index("protocol")
        .reindex(
            [
                protocol[0]
                for protocol
                in EVAL_PROTOCOLS
            ]
        )
    )

    ax.plot(
        subset.index,
        subset["delta_vs_cls_pp"],
        marker="o",
        label=f"top_k={top_k}",
    )

ax.axhline(0.0, linewidth=1)
ax.set_xlabel("Evaluation protocol")
ax.set_ylabel(
    "Gain over frozen CLS "
    "(percentage points)"
)
ax.set_title(
    "Improved residual GATv2: "
    "top-k ablation"
)
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## 12. Plot fixed vs broken decisions


In [ ]:
for protocol_name, _, _ in EVAL_PROTOCOLS:
    subset = (
        topk_only_df[
            topk_only_df["protocol"]
            == protocol_name
        ]
        .sort_values("top_k")
    )

    fig, ax = plt.subplots(figsize=(8, 4))

    x = np.arange(len(subset))
    width = 0.35

    ax.bar(
        x - width / 2,
        subset["fixed_by_model"],
        width,
        label="CLS errors fixed",
    )

    ax.bar(
        x + width / 2,
        subset["broken_by_model"],
        width,
        label="CLS-correct broken",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [
            f"k={int(k)}"
            for k
            in subset["top_k"]
        ]
    )

    ax.set_ylabel("Number of queries")
    ax.set_title(
        f"{protocol_name}: "
        "fixed vs broken CLS decisions"
    )
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    plt.show()


## 13. Save the ablation results

Everything is saved under:

```text
results/topk_ablation_improved_residual_gatv2/
```

The test split is not touched.


In [ ]:
ablation_dir = (
    paths.drive_results_dir
    / "topk_ablation_improved_residual_gatv2"
)
ablation_dir.mkdir(
    parents=True,
    exist_ok=True,
)

training_summary_df.to_csv(
    ablation_dir / "training_summary.csv",
    index=False,
)

comparison_df.to_csv(
    ablation_dir / "evaluation_comparison.csv",
    index=False,
)

topk_only_df.to_csv(
    ablation_dir / "topk_models_only.csv",
    index=False,
)

payload = {
    "top_k_values": TOP_K_VALUES,
    "base_config": BASE_CONFIG,
    "evaluation": {
        "num_episodes": EVAL_NUM_EPISODES,
        "queries_per_class": EVAL_QUERIES_PER_CLASS,
        "seed": EVAL_SEED,
        "protocols": EVAL_PROTOCOLS,
    },
    "edge_diagnostics": edge_diagnostics,
    "training_summary": (
        training_summary_df
        .replace({np.nan: None})
        .to_dict(orient="records")
    ),
    "evaluation_summary": (
        comparison_df
        .replace({np.nan: None})
        .to_dict(orient="records")
    ),
}

atomic_json_save(
    payload,
    ablation_dir / "topk_ablation.json",
)

print("Saved to:")
print(ablation_dir)


## Interpretation

The result we are looking for is not simply "smaller k is better."

The experiment tests the bias–noise tradeoff:

- **too small k** can omit useful correspondences;
- **too large k** can propagate information from weak or accidental patch matches.

A good result would look like:

```text
top_k=3    misses some useful relations
top_k=5    best paired gains / highest net-fixed
top_k=10   more noisy communication
```

But the data may also show that `k=10` is already best. The purpose of this notebook is to establish that empirically while changing only one graph-construction variable.

After choosing `top_k` on validation, freeze that choice before any final test evaluation.
